In [11]:
import pandas as pd

path = r"C:\Users\malee\OneDrive\Desktop\Justin Grad USD\Datasets\Team Project (Tara)\redfin_affordability_monthly_all_states_condo_co_op_homeowner_2023_Jan_to_2026_Jun.csv"

# 1. Verify PROPERTY TYPE column exists
columns = pd.read_csv(path, nrows=0).columns.tolist()
assert "PROPERTY TYPE" in columns, "PROPERTY TYPE column is missing"

# 2–3. Confirm NA is a literal string, and check default pandas
df_as_text = pd.read_csv(path, dtype=str, keep_default_na=False)
literal_na = {
    col: int((df_as_text[col] == "NA").sum())
    for col in df_as_text.columns
    if (df_as_text[col] == "NA").any()
}
print("Literal 'NA' strings by column:", literal_na)

df_default = pd.read_csv(path)  # default na_values already includes "NA"
print("Default-reader nulls:", df_default.isnull().sum().sum())

# 4. Re-read converting NA strings to proper missing values
df = pd.read_csv(
    path,
    na_values=["NA", "N/A", "na", "n/a", ""],
    keep_default_na=True,
)

# 5. Drop unused columns
df = df.drop(columns=["LAST UPDATED", "REGION TYPE"])

# 6. Date-time formatting
df["PERIOD BEGIN"] = pd.to_datetime(df["PERIOD BEGIN"])
df["PERIOD END"] = pd.to_datetime(df["PERIOD END"])

# 7. Seasonality columns from PERIOD BEGIN
df["Year"] = df["PERIOD BEGIN"].dt.year
df["Month"] = df["PERIOD BEGIN"].dt.month
df["Month Name"] = df["PERIOD BEGIN"].dt.month_name()
df["Quarter"] = df["PERIOD BEGIN"].dt.quarter
df["Season"] = df["Month"].map(
    lambda m: "Winter" if m in (12, 1, 2)
    else "Spring" if m in (3, 4, 5)
    else "Summer" if m in (6, 7, 8)
    else "Fall"
)
df["Year-Month"] = df["PERIOD BEGIN"].dt.to_period("M").astype(str)
df["Period Length (Days)"] = (df["PERIOD END"] - df["PERIOD BEGIN"]).dt.days

# 8. Force listed columns to float64
float_cols = [
    "INCOME REQUIRED ($)",
    "YEARS TO SAVE DOWN PAYMENT",
    "MONTHLY MORTGAGE PAYMENT ($)",
    "MONTHLY PROPERTY TAX ($)",
    "MONTHLY PROPERTY INSURANCE ($)",
    "MONTHLY PMI ($)",
    "DOWN PAYMENT AMOUNT ($)",
    "MEDIAN HOUSEHOLD INCOME ($)",
    "MEDIAN SALE PRICE ($)",
    "HOMES SOLD",
]
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

print(df.dtypes)
print(df.shape)
print(df.head(10))
df.to_csv(r"C:\Users\malee\OneDrive\Desktop\Justin Grad USD\Datasets\Team Project (Tara)\Cleaned Data sets\redfin_affordability_condo_cleaned_test.csv", index=False)




Literal 'NA' strings by column: {'INCOME REQUIRED ($)': 210, 'INCOME REQUIRED YOY (%)': 210, 'SHARE OF INCOME ON HOUSING (%)': 210, 'SHARE OF INCOME ON HOUSING YOY (PPTS)': 210, 'AFFORDABLE LISTING SHARE (%)': 210, 'AFFORDABLE LISTING SHARE YOY (PPTS)': 210, 'YEARS TO SAVE DOWN PAYMENT': 210, 'YEARS TO SAVE DOWN PAYMENT YOY (%)': 210, 'MONTHLY HOUSING PAYMENT ($)': 210, 'MONTHLY MORTGAGE PAYMENT ($)': 210, 'MONTHLY PROPERTY TAX ($)': 210, 'MONTHLY PROPERTY INSURANCE ($)': 210, 'MONTHLY PMI ($)': 210, 'DOWN PAYMENT AMOUNT ($)': 210, 'MEDIAN SALE PRICE ($)': 210, 'MEDIAN SALE PRICE YOY (%)': 210, 'HOMES SOLD': 210, 'HOMES SOLD YOY (%)': 210}
Default-reader nulls: 3780
FREQUENCY                                           str
PERIOD BEGIN                             datetime64[us]
PERIOD END                               datetime64[us]
REGION NAME                                         str
PROPERTY TYPE                                       str
INCOME GROUP                                 